# Tally held-out benchmark (open models vs v7)

4-bit inference of open models on the same 126 held-out chunks (5172, 5432, 4329), same v7 full-extraction prompt, greedy decoding, `max_new_tokens=900`, and the same merge/scorer as `renhuang01/tally-heldout`.

Qwen3 thinking is disabled (`enable_thinking=False`) so the comparison stays fair. One model per kernel version to stay inside the 12-hour limit. Change `RUN_MODELS` below between versions.

Version 1 failed: `apply_chat_template(messages=...)` TypeError on transformers 5.20. Version 2 built the template but a diagnostic `decode` aborted after a successful 4-bit load. This version renders the template to a string (`tokenize=False`, `enable_thinking=False`) then tokenizes, and does not abort on the template check. Still Qwen3-8B only; Gemma 3 12B IT follows in the next version.


In [ ]:
# === 1. SETUP ===
# Which models this kernel version should run. One model per version.
# Version 1: Qwen/Qwen3-8B only (priority). Gemma and 30B-A3B are later versions.
RUN_MODELS = ["google/gemma-3-12b-it"]

import os
from pathlib import Path

# Do not pin CUDA_VISIBLE_DEVICES before import: 30B-A3B would need both T4s.
# Single-GPU models are placed on cuda:0 via device_map.
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

os.environ["HF_HOME"] = "/kaggle/tmp/hf"
os.environ["HF_HUB_CACHE"] = "/kaggle/tmp/hf"
os.environ["TRANSFORMERS_CACHE"] = "/kaggle/tmp/hf"
os.makedirs("/kaggle/tmp/hf", exist_ok=True)
os.makedirs("/kaggle/working/heldout", exist_ok=True)
os.makedirs("/kaggle/working/tally", exist_ok=True)

# Do not reinstall torch. transformers>=5 matches the held-out notebook.
%pip install -q bitsandbytes peft "transformers>=5"
# Held-out v1 failed when PEFT rejected the preinstalled torchao.
%pip uninstall -y -q torchao

from huggingface_hub import login


def get_hf_token():
    # 1) the notebook's HF_TOKEN Kaggle secret; 2) a private attached dataset
    # file hf_token.txt (renhuang01/tally-hf-token). The token is never printed.
    try:
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret("HF_TOKEN")
        if tok and tok.strip():
            return tok.strip(), "Kaggle secret"
    except Exception as exc:
        print("HF_TOKEN secret not available:", type(exc).__name__)
    for f in sorted(Path("/kaggle/input").rglob("hf_token.txt")):
        tok = f.read_text(encoding="utf-8").strip()
        if tok:
            return tok, "private dataset file"
    raise RuntimeError("No Hugging Face token: attach the HF_TOKEN secret or the private tally-hf-token dataset.")


_hf_tok, _hf_src = get_hf_token()
login(_hf_tok)
os.environ["HF_TOKEN"] = _hf_tok
print("Logged in to Hugging Face via", _hf_src)
del _hf_tok

import shutil
import torch

print("disk /kaggle/tmp  :", shutil.disk_usage("/kaggle/tmp"))
print("disk /kaggle/working:", shutil.disk_usage("/kaggle/working"))

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}  "
              f"{torch.cuda.get_device_properties(i).total_memory / 1024**3:.1f} GB")
else:
    print("WARNING: no GPU visible. Turn on Settings → Accelerator → GPU T4 x2 and restart.")

print("cuda:", torch.cuda.is_available(), "n_gpu:", torch.cuda.device_count(), "torch", torch.__version__)
import importlib.metadata as _md
for _p in ("transformers", "trl", "peft", "datasets", "bitsandbytes", "accelerate"):
    try:
        print(f"{_p}=={_md.version(_p)}")
    except Exception:
        print(_p, "not installed")


In [ ]:
import json, sys, time, traceback, gc, inspect
from pathlib import Path
from collections import defaultdict
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

HF_CACHE = "/kaggle/tmp/hf"
RESULTS_DIR = Path("/kaggle/working/heldout")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

hits = sorted(Path("/kaggle/input").rglob("heldout_messages.jsonl"))
assert hits, "held-out data not attached"
DATA_DIR = hits[0].parent
sys.path.insert(0, str(DATA_DIR))

def load_jsonl(p):
    return [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]

score_rows = load_jsonl(DATA_DIR / "heldout_messages.jsonl")
score_meta = load_jsonl(DATA_DIR / "heldout_meta.jsonl")
answer_key = json.loads((DATA_DIR / "answer_key_heldout.json").read_text(encoding="utf-8"))
from tally_merge import FIELDS, merge_notes
from score_extractions import score_documents, format_summary_table
print("DATA_DIR", DATA_DIR, "| chunks", len(score_rows), "| docs", sorted(answer_key))
print("RUN_MODELS", RUN_MODELS)

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def slug(mid):
    return mid.split("/")[-1].lower().replace("_", "-")


In [ ]:
# === HELPERS: generation, parsing, scoring (same as held-out, model-agnostic chat template) ===

import re
from collections import defaultdict


def parse_json(text):
    # Parse the FIRST complete JSON object. Same parser as the held-out kernel.
    start = text.find("{")
    if start == -1:
        raise json.JSONDecodeError("no JSON object found", text, 0)
    try:
        obj, _ = json.JSONDecoder().raw_decode(text[start:])
        return obj
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, re.S)
        return json.loads(match.group(0))


def apply_chat(tok, prompt):
    """Build generate() inputs. Disable Qwen3 thinking when the tokenizer supports it.

    Render the template to a string (tokenize=False), then tokenize ourselves.
    This avoids transformers 5.20 BatchEncoding/Encoding quirks (v1: messages=
    keyword TypeError; v2: decode of template output failed). Conversation is
    passed positionally. enable_thinking=False is requested for Qwen3 fairness.
    """
    turns = [{"role": "user", "content": prompt}]
    part_turns = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    thinking = None
    text = None
    last_exc = None
    for conv in (turns, part_turns):
        try:
            text = tok.apply_chat_template(
                conv, tokenize=False, add_generation_prompt=True, enable_thinking=False
            )
            thinking = False
            break
        except TypeError as exc:
            last_exc = exc
            try:
                text = tok.apply_chat_template(
                    conv, tokenize=False, add_generation_prompt=True
                )
                thinking = "unsupported"
                break
            except Exception as exc2:
                last_exc = exc2
                continue
    if text is None:
        raise last_exc
    # Template already has special tokens; do not add them again.
    inputs = tok(text, return_tensors="pt", add_special_tokens=False)
    if not isinstance(inputs, dict):
        inputs = dict(inputs)
    return inputs, thinking, text


def move_to_model(inputs, model):
    try:
        device = model.device
    except Exception:
        device = next(model.parameters()).device
    return {k: v.to(device) if hasattr(v, "to") else v for k, v in inputs.items()}


def generate_reply(model, tok, prompt, max_new_tokens=900):
    inputs, thinking, _text = apply_chat(tok, prompt)
    inputs = move_to_model(inputs, model)
    eos_id = tok.eos_token_id
    pad_id = tok.pad_token_id if tok.pad_token_id is not None else eos_id
    gen_kwargs = dict(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=pad_id,
    )
    if eos_id is not None:
        gen_kwargs["eos_token_id"] = eos_id
    with torch.no_grad():
        output = model.generate(**gen_kwargs)
    reply_ids = output[0][inputs["input_ids"].shape[1]:]
    text = tok.decode(reply_ids, skip_special_tokens=True)
    return text, thinking


def extract_chunk_text(user_content):
    marker = "\nPages "
    pos = user_content.find(marker)
    if pos == -1:
        return user_content
    rest = user_content[pos + 1:]
    nl = rest.find("\n")
    return rest[nl + 1:].lstrip() if nl != -1 else rest


def run_val_extraction(model, tok, label):
    if not score_rows or not score_meta:
        raise RuntimeError("held-out score set is empty or missing")
    per_doc_notes = defaultdict(list)
    per_doc_text = defaultdict(list)
    raw_rows = []
    thinking_flags = []
    t_run = time.time()
    for i, (row, meta) in enumerate(zip(score_rows, score_meta)):
        doc = meta["doc"]
        prompt = row["messages"][0]["content"]
        t0 = time.time()
        print(f"[{label}] {i + 1}/{len(score_rows)}  {doc} chunk {meta.get('chunk_index')}", flush=True)
        thinking = None
        try:
            reply, thinking = generate_reply(model, tok, prompt)
            parsed = parse_json(reply)
            err = None
        except Exception as exc:
            reply = locals().get("reply", "")
            parsed = {"_error": f"{type(exc).__name__}: {exc}", "raw": reply[:2000]}
            err = str(exc)
            print("  parse/generate error:", type(exc).__name__, flush=True)
        thinking_flags.append(thinking)
        dt = time.time() - t0
        per_doc_notes[doc].append(parsed)
        per_doc_text[doc].append(extract_chunk_text(prompt))
        raw_rows.append({
            "doc": doc,
            "chunk_index": meta.get("chunk_index"),
            "reply": reply,
            "parsed": parsed,
            "error": err,
            "seconds": round(dt, 2),
        })
        if (i + 1) % 5 == 0 or i == 0:
            elapsed = time.time() - t_run
            rate = elapsed / (i + 1)
            eta = rate * (len(score_rows) - i - 1)
            print(f"  {dt:.1f}s this chunk | elapsed {elapsed/60:.1f} min | eta {eta/60:.1f} min", flush=True)
        if torch.cuda.is_available() and (i + 1) % 4 == 0:
            torch.cuda.empty_cache()

    merged_by_doc = {}
    for doc, notes in per_doc_notes.items():
        doc_text = "\n\n".join(per_doc_text[doc])
        merged_by_doc[doc] = merge_notes(notes, doc_text=doc_text)

    scores = score_documents(merged_by_doc, answer_key, docs=sorted(merged_by_doc))
    print()
    print(format_summary_table(scores, f"{label} — HELD-OUT (5172, 5432, 4329)"))
    out = {
        "label": label,
        "model_id": None,
        "quantization": "4bit-nf4",
        "decoding": {"do_sample": False, "max_new_tokens": 900},
        "enable_thinking": False if thinking_flags and all(f is False for f in thinking_flags) else (
            "unsupported" if thinking_flags and all(f == "unsupported" for f in thinking_flags) else thinking_flags[:3]
        ),
        "n_chunks": len(score_rows),
        "runtime_sec": time.time() - t_run,
        "merged": merged_by_doc,
        "scores": scores,
        "raw": raw_rows,
    }
    path = RESULTS_DIR / f"{label}_heldout_scores.json"
    path.write_text(json.dumps(out, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
    print("wrote", path)
    return out


def is_gated_403(exc):
    name = type(exc).__name__
    msg = str(exc).lower()
    return (
        "GatedRepo" in name
        or "403" in str(exc)
        or "gated repo" in msg
        or "not in the authorized list" in msg
        or "access to model" in msg and "restricted" in msg
    )


def load_model_4bit(mid, use_multi_gpu=False):
    tok = AutoTokenizer.from_pretrained(mid, cache_dir=HF_CACHE)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,  # T4 has no bf16
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )
    load_kw = dict(
        quantization_config=bnb,
        device_map="auto" if use_multi_gpu else {"": 0},
        cache_dir=HF_CACHE,
        dtype=torch.float16,
    )
    try:
        model = AutoModelForCausalLM.from_pretrained(mid, **load_kw)
    except TypeError:
        load_kw.pop("dtype", None)
        load_kw["torch_dtype"] = torch.float16
        try:
            model = AutoModelForCausalLM.from_pretrained(mid, **load_kw)
        except Exception:
            # Gemma 3 is image-text; fall back to the auto multimodal class.
            from transformers import AutoModelForImageTextToText
            model = AutoModelForImageTextToText.from_pretrained(mid, **load_kw)
    except Exception as exc:
        if "image-text" in str(exc).lower() or "Gemma3" in str(exc) or "AutoModelForCausalLM" in str(exc):
            from transformers import AutoModelForImageTextToText
            try:
                model = AutoModelForImageTextToText.from_pretrained(mid, **load_kw)
            except Exception:
                raise exc
        else:
            raise
    model.eval()
    if hasattr(model, "config"):
        model.config.use_cache = True
    # Confirm the thinking flag on a dummy turn (no document text printed).
    # Never abort a successful load because of this check (v2 did).
    thinking = None
    try:
        _dummy, thinking, text = apply_chat(tok, "Reply with JSON only: {\"ok\": true}")
        has_think = "<think>" in (text or "")
        print(
            f"loaded {mid} 4-bit | device_map={'auto' if use_multi_gpu else '{0}'} | "
            f"thinking_flag={thinking} | '<think>' in template={has_think}",
            flush=True,
        )
        del _dummy, text
    except Exception as exc:
        print(
            f"loaded {mid} 4-bit | device_map={'auto' if use_multi_gpu else '{0}'} | "
            f"template check skipped: {type(exc).__name__}",
            flush=True,
        )
        thinking = "check_failed"
    return tok, model, thinking


In [ ]:
# === RUN ===
# Qwen3-8B: 4-bit ~5 GB, one T4. 30B-A3B would use both T4s and is not in this version.
# Gemma-3-12B-IT is gated; a later version will retry and skip on 403.

SKIP_LOG = []
RESULTS = {}

def should_use_multi_gpu(mid):
    name = mid.lower()
    return "30b" in name or "a3b" in name


def disk_free_gb(path="/kaggle/tmp"):
    return shutil.disk_usage(path).free / (1024 ** 3)


for mid in RUN_MODELS:
    label = slug(mid)
    print("=" * 72)
    print("START", mid, "label=", label, flush=True)
    # Guard: 30B official weights are ~61 GB bf16. Skip if tmp cannot hold them.
    if "30b" in mid.lower() and disk_free_gb() < 70:
        msg = f"SKIP {mid}: /kaggle/tmp has only {disk_free_gb():.1f} GB free, official weights are ~61 GB"
        print(msg, flush=True)
        SKIP_LOG.append({"model": mid, "reason": msg})
        continue
    t0 = time.time()
    tok = model = out = None
    try:
        tok, model, thinking = load_model_4bit(mid, use_multi_gpu=should_use_multi_gpu(mid))
        out = run_val_extraction(model, tok, label)
        out["model_id"] = mid
        out["load_and_run_sec"] = time.time() - t0
        out["enable_thinking_requested"] = False
        # rewrite with model_id
        path = RESULTS_DIR / f"{label}_heldout_scores.json"
        path.write_text(json.dumps(out, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
        RESULTS[label] = out
        o = out["scores"]["overall"]
        print(
            f"DONE {mid} in {(time.time()-t0)/60:.1f} min  "
            f"stated={o['stated_correct']}/{o['stated_n']} ({o['stated_accuracy']:.1%})  "
            f"hallucinated={o['hallucinated']}",
            flush=True,
        )
    except Exception as exc:
        traceback.print_exc()
        if is_gated_403(exc):
            msg = f"SKIP {mid}: gated 403 (license not yet accepted on this HF account)"
        else:
            msg = f"FAIL {mid}: {type(exc).__name__}"
        print(msg, flush=True)
        SKIP_LOG.append({"model": mid, "reason": msg, "exc_type": type(exc).__name__})
    finally:
        model = None
        tok = None
        free_gpu()
        print(f"{mid} released, elapsed {(time.time()-t0)/60:.1f} min", flush=True)


In [ ]:
# === SUMMARY ===

def overall_line(name, s):
    o = s["overall"]
    return (
        f"{name:<22} stated={o['stated_accuracy']:.1%} ({o['stated_correct']}/{o['stated_n']})  "
        f"acc+abs={o['accuracy_incl_abstention']:.1%}  "
        f"correct={o['correct']} wrong={o['wrong']} conflict={o['conflict-unresolved']} "
        f"abstained={o['correctly-abstained']} hallucinated={o['hallucinated']}"
    )

comparison = {
    "heldout_docs": sorted(answer_key),
    "n_chunks": len(score_rows),
    "quantization": "4bit-nf4",
    "decoding": {"do_sample": False, "max_new_tokens": 900},
    "enable_thinking": False,
    "run_models": RUN_MODELS,
    "skipped": SKIP_LOG,
    "models": {},
}
for label, out in RESULTS.items():
    print(overall_line(label, out["scores"]))
    print(format_summary_table(out["scores"], f"{label} — held-out"))
    comparison["models"][label] = {
        "model_id": out.get("model_id"),
        "scores": out["scores"],
        "merged": out["merged"],
        "runtime_sec": out.get("runtime_sec"),
        "load_and_run_sec": out.get("load_and_run_sec"),
        "enable_thinking": out.get("enable_thinking"),
        "quantization": out.get("quantization"),
        "decoding": out.get("decoding"),
    }

if SKIP_LOG:
    print("SKIPPED:")
    for row in SKIP_LOG:
        print(" -", row)

# Do not print extracted values or gold values.
for p in (RESULTS_DIR / "bench_comparison.json", Path("/kaggle/working/bench_comparison.json")):
    p.write_text(json.dumps(comparison, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
    print("wrote", p)

notes = {
    "run_models": RUN_MODELS,
    "skipped": SKIP_LOG,
    "completed": {
        label: {
            "model_id": out.get("model_id"),
            "stated_correct": out["scores"]["overall"]["stated_correct"],
            "stated_n": out["scores"]["overall"]["stated_n"],
            "stated_accuracy": out["scores"]["overall"]["stated_accuracy"],
            "hallucinated": out["scores"]["overall"]["hallucinated"],
            "runtime_sec": out.get("runtime_sec"),
            "load_and_run_sec": out.get("load_and_run_sec"),
            "enable_thinking": out.get("enable_thinking"),
        }
        for label, out in RESULTS.items()
    },
    "notes": (
        "4-bit NF4, greedy, max_new_tokens=900, same v7 prompt and scorer. "
        "Qwen3 enable_thinking=False. Token never printed."
    ),
}
Path("/kaggle/working/bench_run_notes.json").write_text(
    json.dumps(notes, indent=2, ensure_ascii=False, default=str), encoding="utf-8"
)
print("wrote /kaggle/working/bench_run_notes.json")
print("ALL DONE")
